# Notebook 5 — Feature Engineering

## Objective

Create machine-learning features from the training, validation, and test
datasets based on the findings from Notebook 4.

All preprocessing transformations will be fitted on the training data only
and then applied unchanged to the validation and test data.

## Inputs

- train.parquet
- validation.parquet
- test.parquet

## Outputs

- Transformed training, validation, and test features
- Target datasets
- Fitted preprocessing pipeline
- Feature list

In [13]:
import pandas as pd
import numpy as np
from pathlib import Path
import joblib

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

In [19]:
from pathlib import Path

data_path = Path("../data")

print(data_path.resolve())

C:\Users\user\Desktop\MLOps-Olist\data


In [20]:
train_df = pd.read_parquet(data_path / "train.parquet")
val_df = pd.read_parquet(data_path / "validation.parquet")
test_df = pd.read_parquet(data_path / "test.parquet")

In [21]:
print("Train shape:", train_df.shape)
print("Validation shape:", val_df.shape)
print("Test shape:", test_df.shape)

Train shape: (67533, 29)
Validation shape: (14471, 29)
Test shape: (14472, 29)


In [22]:
target = "late"

print("Target column:", target)
print("Target dtype:", train_df[target].dtype)

Target column: late
Target dtype: int64


In [23]:
print("Columns:")
for i, column in enumerate(train_df.columns, start=1):
    print(f"{i:02d}. {column}")

Columns:
01. order_id
02. customer_id
03. order_status
04. order_purchase_timestamp
05. order_approved_at
06. order_delivered_carrier_date
07. order_delivered_customer_date
08. order_estimated_delivery_date
09. item_count
10. total_item_price
11. total_freight_value
12. avg_item_price
13. payment_count
14. total_payment_value
15. max_payment_installments
16. review_count
17. avg_review_score
18. min_review_score
19. max_review_score
20. customer_unique_id
21. customer_zip_code_prefix
22. customer_city
23. customer_state
24. unique_product_count
25. unique_seller_count
26. unique_category_count
27. avg_latitude
28. avg_longitude
29. late


## Feature Selection Strategy


In [24]:
target = "late"

print("Target column:", target)
print("Target dtype:", train_df[target].dtype)

Target column: late
Target dtype: int64


## Identifier and Target-Leakage Features


In [26]:
id_columns = [
    "order_id",
    "customer_id",
    "customer_unique_id"
]

leakage_columns = [
    "order_delivered_carrier_date",
    "order_delivered_customer_date"
]

drop_columns = id_columns + leakage_columns + [target]

print("ID columns:")
print(id_columns)

print("\nPotential leakage columns:")
print(leakage_columns)

print("\nColumns excluded from features:")
print(drop_columns)

ID columns:
['order_id', 'customer_id', 'customer_unique_id']

Potential leakage columns:
['order_delivered_carrier_date', 'order_delivered_customer_date']

Columns excluded from features:
['order_id', 'customer_id', 'customer_unique_id', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'late']


## Temporal Feature Engineering

In [27]:
def create_time_features(df):
    df = df.copy()

    purchase_time = pd.to_datetime(
        df["order_purchase_timestamp"],
        errors="coerce"
    )

    approved_time = pd.to_datetime(
        df["order_approved_at"],
        errors="coerce"
    )

    estimated_delivery = pd.to_datetime(
        df["order_estimated_delivery_date"],
        errors="coerce"
    )

    df["purchase_year"] = purchase_time.dt.year
    df["purchase_month"] = purchase_time.dt.month
    df["purchase_dayofweek"] = purchase_time.dt.dayofweek
    df["purchase_hour"] = purchase_time.dt.hour

    df["approval_delay_hours"] = (
        (approved_time - purchase_time).dt.total_seconds() / 3600
    )

    df["estimated_delivery_days_from_purchase"] = (
        (estimated_delivery - purchase_time).dt.total_seconds() / 86400
    )

    return df

In [28]:
train_fe = create_time_features(train_df)
val_fe = create_time_features(val_df)
test_fe = create_time_features(test_df)

print("Train:", train_fe.shape)
print("Validation:", val_fe.shape)
print("Test:", test_fe.shape)

Train: (67533, 35)
Validation: (14471, 35)
Test: (14472, 35)


## Engineered Temporal Features


In [29]:
temporal_features = [
    "purchase_year",
    "purchase_month",
    "purchase_dayofweek",
    "purchase_hour",
    "approval_delay_hours",
    "estimated_delivery_days_from_purchase"
]

train_fe[temporal_features].describe().T

,count,mean,std,min,25%,50%,75%,max
purchase_year,67533.0,2017.348911,0.485006,2016.000000,2017.000000,2017.000000,2018.000000,2018.000000
purchase_month,67533.0,5.971629,3.756504,1.000000,3.000000,5.000000,10.000000,12.000000
purchase_dayofweek,67533.0,2.789777,1.967275,0.000000,1.000000,3.000000,4.000000,6.000000
purchase_hour,67533.0,14.779486,5.353837,0.000000,11.000000,15.000000,19.000000,23.000000
approval_delay_hours,67519.0,9.931328,20.420540,0.000000,0.205278,0.307500,13.448472,741.443611
estimated_delivery_days_from_purchase,67533.0,24.597288,8.022133,7.005127,19.543333,23.621377,28.565984,155.135463


## Final Feature Candidate Set


In [30]:
raw_datetime_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_estimated_delivery_date"
]

final_drop_columns = (
    drop_columns
    + raw_datetime_columns
)

X_train = train_fe.drop(columns=final_drop_columns)
X_val = val_fe.drop(columns=final_drop_columns)
X_test = test_fe.drop(columns=final_drop_columns)

y_train = train_fe[target]
y_val = val_fe[target]
y_test = test_fe[target]

print("X_train shape:", X_train.shape)
print("X_val shape:", X_val.shape)
print("X_test shape:", X_test.shape)

X_train shape: (67533, 26)
X_val shape: (14471, 26)
X_test shape: (14472, 26)


## Target Separation Check


In [31]:
assert target not in X_train.columns
assert target not in X_val.columns
assert target not in X_test.columns

print("Target leakage check passed.")
print(f"'{target}' is not present in the feature matrices.")

Target leakage check passed.
'late' is not present in the feature matrices.


## Candidate Machine-Learning Features


In [32]:
print("Number of candidate features:", X_train.shape[1])

for i, column in enumerate(X_train.columns, start=1):
    print(f"{i:02d}. {column}")

Number of candidate features: 26
01. order_status
02. item_count
03. total_item_price
04. total_freight_value
05. avg_item_price
06. payment_count
07. total_payment_value
08. max_payment_installments
09. review_count
10. avg_review_score
11. min_review_score
12. max_review_score
13. customer_zip_code_prefix
14. customer_city
15. customer_state
16. unique_product_count
17. unique_seller_count
18. unique_category_count
19. avg_latitude
20. avg_longitude
21. purchase_year
22. purchase_month
23. purchase_dayofweek
24. purchase_hour
25. approval_delay_hours
26. estimated_delivery_days_from_purchase


In [33]:
print("Number of candidate features:", X_train.shape[1])

for i, column in enumerate(X_train.columns, start=1):
    print(f"{i:02d}. {column}")

Number of candidate features: 26
01. order_status
02. item_count
03. total_item_price
04. total_freight_value
05. avg_item_price
06. payment_count
07. total_payment_value
08. max_payment_installments
09. review_count
10. avg_review_score
11. min_review_score
12. max_review_score
13. customer_zip_code_prefix
14. customer_city
15. customer_state
16. unique_product_count
17. unique_seller_count
18. unique_category_count
19. avg_latitude
20. avg_longitude
21. purchase_year
22. purchase_month
23. purchase_dayofweek
24. purchase_hour
25. approval_delay_hours
26. estimated_delivery_days_from_purchase


## Prediction-Time Feature Availability


In [34]:
post_prediction_columns = [
    "review_count",
    "avg_review_score",
    "min_review_score",
    "max_review_score",
    "order_status"
]

X_train = X_train.drop(columns=post_prediction_columns)
X_val = X_val.drop(columns=post_prediction_columns)
X_test = X_test.drop(columns=post_prediction_columns)

print("Removed columns:")
print(post_prediction_columns)

print("\nRemaining feature count:", X_train.shape[1])

Removed columns:
['review_count', 'avg_review_score', 'min_review_score', 'max_review_score', 'order_status']

Remaining feature count: 21


## Missing Value Analysis


In [35]:
missing_summary = (
    X_train.isna()
    .sum()
    .to_frame("missing_count")
)

missing_summary["missing_percentage"] = (
    missing_summary["missing_count"]
    / len(X_train)
    * 100
)

missing_summary = (
    missing_summary[
        missing_summary["missing_count"] > 0
    ]
    .sort_values("missing_count", ascending=False)
)

missing_summary

,missing_count,missing_percentage
avg_longitude,181,0.268017
avg_latitude,181,0.268017
approval_delay_hours,14,0.020731
payment_count,1,0.001481
total_payment_value,1,0.001481
max_payment_installments,1,0.001481


## Numerical Features


In [36]:
numeric_features = [
    "item_count",
    "total_item_price",
    "total_freight_value",
    "avg_item_price",
    "payment_count",
    "total_payment_value",
    "max_payment_installments",
    "unique_product_count",
    "unique_seller_count",
    "unique_category_count",
    "avg_latitude",
    "avg_longitude",
    "purchase_year",
    "purchase_month",
    "purchase_dayofweek",
    "purchase_hour",
    "approval_delay_hours",
    "estimated_delivery_days_from_purchase"
]

print("Number of numerical features:", len(numeric_features))

for i, feature in enumerate(numeric_features, start=1):
    print(f"{i:02d}. {feature}")

Number of numerical features: 18
01. item_count
02. total_item_price
03. total_freight_value
04. avg_item_price
05. payment_count
06. total_payment_value
07. max_payment_installments
08. unique_product_count
09. unique_seller_count
10. unique_category_count
11. avg_latitude
12. avg_longitude
13. purchase_year
14. purchase_month
15. purchase_dayofweek
16. purchase_hour
17. approval_delay_hours
18. estimated_delivery_days_from_purchase


## Categorical Features


In [37]:
categorical_features = [
    "order_status",
    "customer_zip_code_prefix",
    "customer_city",
    "customer_state"
]

In [38]:
categorical_features = [
    "customer_zip_code_prefix",
    "customer_city",
    "customer_state"
]

print("Number of categorical features:", len(categorical_features))

for i, feature in enumerate(categorical_features, start=1):
    print(f"{i:02d}. {feature}")

Number of categorical features: 3
01. customer_zip_code_prefix
02. customer_city
03. customer_state


## Feature Type Consistency Check


In [39]:
all_selected_features = numeric_features + categorical_features

print("X_train features:", len(X_train.columns))
print("Selected features:", len(all_selected_features))

missing_from_selection = set(X_train.columns) - set(all_selected_features)
extra_in_selection = set(all_selected_features) - set(X_train.columns)

print("\nFeatures not assigned to a preprocessing group:")
print(missing_from_selection)

print("\nFeatures not present in X_train:")
print(extra_in_selection) 

X_train features: 21
Selected features: 21

Features not assigned to a preprocessing group:
set()

Features not present in X_train:
set()


## Numerical Feature Preprocessing


In [40]:
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

print("Numerical preprocessing pipeline created.")

Numerical preprocessing pipeline created.


## Categorical Feature Preprocessing


In [41]:
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    (
        "encoder",
        OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=True
        )
    )
])

print("Categorical preprocessing pipeline created.")

Categorical preprocessing pipeline created.


## Combined Preprocessing Transformer


In [42]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_pipeline, numeric_features),
        ("cat", categorical_pipeline, categorical_features)
    ]
)

print("ColumnTransformer created successfully.")

ColumnTransformer created successfully.


## Fit Preprocessor on Training Data Only


In [43]:
X_train_transformed = preprocessor.fit_transform(X_train)

print("Training data transformed successfully.")
print("Transformed training shape:", X_train_transformed.shape)

Training data transformed successfully.
Transformed training shape: (67533, 17540)


## Transform Validation and Test Data


In [44]:
X_val_transformed = preprocessor.transform(X_val)
X_test_transformed = preprocessor.transform(X_test)

print("Validation transformed shape:", X_val_transformed.shape)
print("Test transformed shape:", X_test_transformed.shape)

Validation transformed shape: (14471, 17540)
Test transformed shape: (14472, 17540)


## Transformed Feature Consistency Check


In [45]:
print("Train transformed:", X_train_transformed.shape)
print("Validation transformed:", X_val_transformed.shape)
print("Test transformed:", X_test_transformed.shape)

assert X_train_transformed.shape[1] == X_val_transformed.shape[1]
assert X_train_transformed.shape[1] == X_test_transformed.shape[1]

print("\nFeature dimension consistency check passed.")

Train transformed: (67533, 17540)
Validation transformed: (14471, 17540)
Test transformed: (14472, 17540)

Feature dimension consistency check passed.


## Transformed Feature Names


In [46]:
feature_names = preprocessor.get_feature_names_out()

print("Number of transformed features:", len(feature_names))
print("\nFirst 20 transformed features:")

for i, feature in enumerate(feature_names[:20], start=1):
    print(f"{i:02d}. {feature}")

Number of transformed features: 17540

First 20 transformed features:
01. num__item_count
02. num__total_item_price
03. num__total_freight_value
04. num__avg_item_price
05. num__payment_count
06. num__total_payment_value
07. num__max_payment_installments
08. num__unique_product_count
09. num__unique_seller_count
10. num__unique_category_count
11. num__avg_latitude
12. num__avg_longitude
13. num__purchase_year
14. num__purchase_month
15. num__purchase_dayofweek
16. num__purchase_hour
17. num__approval_delay_hours
18. num__estimated_delivery_days_from_purchase
19. cat__customer_zip_code_prefix_1004
20. cat__customer_zip_code_prefix_1005


## Post-Preprocessing Missing Value Check


In [48]:
from scipy import sparse

print("Train transformed contains NaN:", np.isnan(X_train_transformed.data).any())
print("Validation transformed contains NaN:", np.isnan(X_val_transformed.data).any())
print("Test transformed contains NaN:", np.isnan(X_test_transformed.data).any())

Train transformed contains NaN: False
Validation transformed contains NaN: False
Test transformed contains NaN: False


## Save Transformed Feature Matrices


In [49]:
from scipy import sparse

output_path = data_path / "features"
output_path.mkdir(parents=True, exist_ok=True)

sparse.save_npz(
    output_path / "X_train.npz",
    X_train_transformed
)

sparse.save_npz(
    output_path / "X_validation.npz",
    X_val_transformed
)

sparse.save_npz(
    output_path / "X_test.npz",
    X_test_transformed
)

print("Transformed feature matrices saved successfully.")
print("Output directory:", output_path.resolve())

Transformed feature matrices saved successfully.
Output directory: C:\Users\user\Desktop\MLOps-Olist\data\features


## Save Target Variables


In [50]:
y_train.to_frame().to_parquet(
    output_path / "y_train.parquet",
    index=False
)

y_val.to_frame().to_parquet(
    output_path / "y_validation.parquet",
    index=False
)

y_test.to_frame().to_parquet(
    output_path / "y_test.parquet",
    index=False
)

print("Target files saved successfully.") 

Target files saved successfully.


## Save Fitted Preprocessing Pipeline


In [51]:
preprocessor_path = output_path / "preprocessor.joblib"

joblib.dump(
    preprocessor,
    preprocessor_path
)

print("Preprocessor saved successfully.")
print(preprocessor_path.resolve()) 

Preprocessor saved successfully.
C:\Users\user\Desktop\MLOps-Olist\data\features\preprocessor.joblib


## Save Transformed Feature List


In [52]:
feature_list_path = output_path / "feature_list.txt"

with open(feature_list_path, "w", encoding="utf-8") as f:
    for feature in feature_names:
        f.write(feature + "\n")

print("Feature list saved successfully.")
print(feature_list_path.resolve())

Feature list saved successfully.
C:\Users\user\Desktop\MLOps-Olist\data\features\feature_list.txt


## Final Artifact Validation


In [53]:
expected_artifacts = [
    "X_train.npz",
    "X_validation.npz",
    "X_test.npz",
    "y_train.parquet",
    "y_validation.parquet",
    "y_test.parquet",
    "preprocessor.joblib",
    "feature_list.txt"
]

for artifact in expected_artifacts:
    artifact_path = output_path / artifact
    print(f"{artifact}: {artifact_path.exists()}")

X_train.npz: True
X_validation.npz: True
X_test.npz: True
y_train.parquet: True
y_validation.parquet: True
y_test.parquet: True
preprocessor.joblib: True
feature_list.txt: True


## Notebook 5 Summary

The feature engineering and preprocessing pipeline has been completed.

The final feature set contains numerical and categorical variables based
on prediction-time availability and the findings from exploratory analysis.

Numerical features were imputed and standardized, while categorical
features were imputed and one-hot encoded.

The preprocessing pipeline was fitted using training data only and then
applied unchanged to validation and test data.

The resulting feature matrices, target variables, preprocessing pipeline,
and transformed feature names were saved as reusable artifacts.

In [54]:
print("Notebook 5 — Feature Engineering completed successfully.")
print()
print("Train features:", X_train_transformed.shape)
print("Validation features:", X_val_transformed.shape)
print("Test features:", X_test_transformed.shape)
print("Number of transformed features:", len(feature_names))
print()
print("Artifacts saved in:")
print(output_path.resolve())

Notebook 5 — Feature Engineering completed successfully.

Train features: (67533, 17540)
Validation features: (14471, 17540)
Test features: (14472, 17540)
Number of transformed features: 17540

Artifacts saved in:
C:\Users\user\Desktop\MLOps-Olist\data\features


In [55]:
from scipy import sparse

print("Train transformed contains NaN:", np.isnan(X_train_transformed.data).any())
print("Validation transformed contains NaN:", np.isnan(X_val_transformed.data).any())
print("Test transformed contains NaN:", np.isnan(X_test_transformed.data).any())

Train transformed contains NaN: False
Validation transformed contains NaN: False
Test transformed contains NaN: False
